# ZeroMQ: Publish-Subscribe Pattern

Ggfs. Installation von ZeroMQ Package

In [2]:
!pip install pyzmq

Imports

In [3]:
import zmq
import threading
import time
from datetime import datetime

Publisher

In [4]:
def run_publisher():
    context = zmq.Context()
    socket = context.socket(zmq.PUB)
    socket.bind("tcp://*:12345")

    print("🟢 Publisher läuft...")

    # wichtig: kurze Pause, damit Subscriber Zeit zum Verbinden haben
    time.sleep(1)

    for i in range(10):
        current_time = datetime.now().strftime("%H:%M:%S")
        message = f"TIME {current_time}"
        print("📤 Publisher sendet:", message)
        socket.send_string(message)
        time.sleep(1)

    socket.close()
    context.term()
    print("🛑 Publisher beendet")

Subscriber

In [5]:
def run_subscriber(name="Subscriber"):
    context = zmq.Context()
    socket = context.socket(zmq.SUB)
    socket.connect("tcp://localhost:12345")

    # nur Nachrichten mit Topic "TIME" abonnieren
    socket.setsockopt_string(zmq.SUBSCRIBE, "TIME")

    print(f"🔵 {name} wartet auf TIME-Nachrichten...")

    for _ in range(5):
        message = socket.recv_string()
        print(f"📥 {name} empfängt:", message)

    socket.close()
    context.term()
    print(f"🛑 {name} beendet")

Publisher und Subscriber starten.  
Wir müssen Threading verwenden.

In [6]:
subscriber_thread = threading.Thread(target=run_subscriber, args=("Sub1",), daemon=True)
publisher_thread = threading.Thread(target=run_publisher, daemon=True)

subscriber_thread.start()
time.sleep(0.5)
publisher_thread.start()

🔵 Sub1 wartet auf TIME-Nachrichten...
🟢 Publisher läuft...
📤 Publisher sendet: TIME 15:55:10
📥 Sub1 empfängt: TIME 15:55:10
📤 Publisher sendet: TIME 15:55:11
📥 Sub1 empfängt: TIME 15:55:11
📤 Publisher sendet: TIME 15:55:12
📥 Sub1 empfängt: TIME 15:55:12
📤 Publisher sendet: TIME 15:55:13
📥 Sub1 empfängt: TIME 15:55:13
📤 Publisher sendet: TIME 15:55:14
📥 Sub1 empfängt: TIME 15:55:14
🛑 Sub1 beendet
📤 Publisher sendet: TIME 15:55:15
📤 Publisher sendet: TIME 15:55:16
📤 Publisher sendet: TIME 15:55:17
📤 Publisher sendet: TIME 15:55:18
📤 Publisher sendet: TIME 15:55:19
🛑 Publisher beendet


# Auswertung
Bei PUB/SUB gilt:  
- Publisher sendet an viele Empfänger
- Subscriber wählen Topics aus
- lose Kopplung
- asynchron
- Sender und Empfänger müssen nicht gleichzeitig starten
- aber: späte Subscriber verpassen alte Nachrichten

Damit:  
👉 PUB/SUB ist gut für Live-Streams, Events, Sensorwerte, Ticker  
👉 aber nicht gut, wenn jede einzelne Nachricht garantiert ankommen muss  
